# Variational Warmstart Ablation
**Tommaso R. Marena (2026) — Section 5.3**

Measures the speedup from variational warmstart (Contribution 3) on all six real datasets.
For each dataset, records M_cold (samples without warmstart), M_warm (with warmstart),
and the speedup ratio. Produces the per-dataset ablation table for Section 5.3.

**Produces:**
- `warmstart_ablation_table.json` — M_cold, M_warm, speedup per dataset
- `warmstart_ablation.pdf` — bar chart of speedup by dataset
- LaTeX table snippet

> **Runtime:** ~30 min on T4.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q',
    'quantum-oracle-sketching[dev,noise,kernel] @ git+https://github.com/Tommaso-R-Marena/quantum_oracle_sketching.git'],
    capture_output=True)
subprocess.run([sys.executable,'-m','pip','install','-q','scanpy','anndata'], capture_output=True)
print('Installed.')

In [ ]:
import os
MOUNT_DRIVE = True
if MOUNT_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/qos_warmstart'
else:
    OUTPUT_DIR = '/content/qos_warmstart'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import numpy as np, jax, jax.numpy as jnp, json, time
import matplotlib.pyplot as plt, matplotlib
matplotlib.rcParams.update({'figure.dpi': 150, 'font.size': 10, 'font.family': 'serif'})

EPSILON_TARGET  = 0.05   # TVD convergence threshold
M_MAX           = 50000  # max samples before giving up
NUM_FOURIER     = 32     # Fourier modes for warmstart ansatz
WARMSTART_STEPS = 200    # gradient descent steps for warmstart
WARMSTART_LR    = 0.02
NUM_TRIALS      = 3      # random seeds per dataset
SEED            = 42
print(f'Epsilon target: {EPSILON_TARGET}, M_max: {M_MAX}')

## 1. Dataset Loaders & Helpers

In [ ]:
from qos.core.oracle_sketch import q_oracle_sketch_boolean
from qos.theory import VariationalWarmstart

def tvd_diag(diag_noisy, diag_ideal):
    N = len(diag_ideal)
    n = int(np.log2(N))
    H = np.array([[1,1],[1,-1]])/np.sqrt(2)
    Hn = H.copy()
    for _ in range(n-1): Hn = np.kron(Hn, H)
    def probs(d):
        s = Hn @ (np.array(d) * np.ones(N)/np.sqrt(N))
        p = np.abs(s)**2; return p/p.sum()
    return 0.5*float(np.sum(np.abs(probs(diag_noisy)-probs(diag_ideal))))

def find_M_cold(truth_table, epsilon=EPSILON_TARGET, m_max=M_MAX):
    """Binary search for M_cold: smallest M s.t. TVD < epsilon."""
    tt = jnp.array(truth_table, dtype=jnp.float64)
    d_ideal, _ = q_oracle_sketch_boolean(tt, m_max)
    lo, hi = 10, m_max
    while lo < hi - 1:
        mid = (lo + hi) // 2
        d, _ = q_oracle_sketch_boolean(tt, mid)
        if tvd_diag(d, d_ideal) < epsilon: hi = mid
        else: lo = mid
    return hi

def find_M_warm(truth_table, epsilon=EPSILON_TARGET, m_max=M_MAX):
    """M_warm: with variational warmstart."""
    tt = jnp.array(truth_table, dtype=jnp.float64)
    d_ideal, _ = q_oracle_sketch_boolean(tt, m_max)
    vw = VariationalWarmstart(tt, num_fourier_modes=NUM_FOURIER,
                              learning_rate=WARMSTART_LR, num_steps=WARMSTART_STEPS)
    vw.fit(unit_num_samples=2000)
    lo, hi = 10, m_max
    while lo < hi - 1:
        mid = (lo + hi) // 2
        d_warm = vw.predict() * q_oracle_sketch_boolean(tt, mid)[0]
        if tvd_diag(d_warm, d_ideal) < epsilon: hi = mid
        else: lo = mid
    return hi

print('Helpers loaded.')

## 2. Load Datasets & Build Truth Tables

In [ ]:
# Each dataset: load a representative feature vector as the truth table
# (binarised at median) for the warmstart experiment.
# We use the first principal component as a proxy for a Boolean function.
import scipy.sparse as sp
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

def pc1_truth_table(X, N=256):
    """Binarise first principal component at median, truncate/pad to N."""
    if sp.issparse(X): X = X.toarray()
    svd = TruncatedSVD(n_components=1, random_state=SEED)
    pc = svd.fit_transform(X).ravel()
    pc = np.resize(pc, N)  # truncate or tile to exactly N
    return (pc >= np.median(pc)).astype(float)

datasets = {}
N_TT = 256  # truth table length for all datasets

# IMDb
print('Loading IMDb...')
from qos.experiments.real_datasets.imdb.imdb_utils import load_imdb_data
X_raw, _ = load_imdb_data()
vec = TfidfVectorizer(min_df=10, stop_words='english', max_features=N_TT)
X_imdb = vec.fit_transform(X_raw[:5000])
datasets['IMDb'] = pc1_truth_table(X_imdb, N_TT)

# 20 Newsgroups
print('Loading 20 Newsgroups...')
data = fetch_20newsgroups(subset='train', remove=('headers','footers','quotes'))
vec2 = TfidfVectorizer(min_df=10, stop_words='english', max_features=N_TT)
X_news = vec2.fit_transform(data.data[:5000])
datasets['20 Newsgroups'] = pc1_truth_table(X_news, N_TT)

# PBMC3k
print('Loading PBMC3k...')
from qos.experiments.real_datasets.pbmc68k.pbmc_utils import load_pbmc3k
adata3k, _ = load_pbmc3k()
X3k = adata3k.X.toarray() if sp.issparse(adata3k.X) else adata3k.X
datasets['PBMC3k'] = pc1_truth_table(X3k[:, :N_TT], N_TT)

# Dorothea
print('Loading Dorothea...')
from qos.experiments.real_datasets.dorothea.dorothea_utils import load_dorothea
X_dor, _ = load_dorothea()
datasets['Dorothea'] = pc1_truth_table(X_dor, N_TT)

# Splice
print('Loading Splice...')
from qos.experiments.real_datasets.splice.splice_utils import load_splice
X_spl, _ = load_splice()
datasets['Splice'] = pc1_truth_table(X_spl, N_TT)

print(f'Loaded {len(datasets)} datasets, N={N_TT} each.')

## 3. Run Ablation

In [ ]:
ablation_results = {}
t0 = time.time()
rng_ab = np.random.default_rng(SEED)

for name, base_tt in datasets.items():
    print(f'\n{name}:')
    m_colds, m_warms = [], []
    for trial in range(NUM_TRIALS):
        # Perturb truth table slightly per trial
        noise_mask = rng_ab.random(len(base_tt)) < 0.05
        tt = base_tt.copy(); tt[noise_mask] = 1 - tt[noise_mask]
        mc = find_M_cold(tt)
        mw = find_M_warm(tt)
        m_colds.append(mc); m_warms.append(mw)
        print(f'  trial {trial+1}: M_cold={mc}, M_warm={mw}, speedup={mc/max(mw,1):.1f}x')
    ablation_results[name] = {
        'M_cold_mean':  float(np.mean(m_colds)),
        'M_warm_mean':  float(np.mean(m_warms)),
        'M_cold_std':   float(np.std(m_colds)),
        'M_warm_std':   float(np.std(m_warms)),
        'speedup_mean': float(np.mean(m_colds) / max(np.mean(m_warms), 1)),
        'epsilon':      EPSILON_TARGET,
    }
    print(f'  => M_cold={np.mean(m_colds):.0f}±{np.std(m_colds):.0f}  M_warm={np.mean(m_warms):.0f}±{np.std(m_warms):.0f}  speedup={ablation_results[name]["speedup_mean"]:.1f}x')

print(f'\nTotal time: {time.time()-t0:.0f}s')
with open(os.path.join(OUTPUT_DIR,'warmstart_ablation_table.json'),'w') as f:
    json.dump(ablation_results, f, indent=2)


In [ ]:
names   = list(ablation_results.keys())
speedups = [ablation_results[n]['speedup_mean'] for n in names]
m_cold  = [ablation_results[n]['M_cold_mean'] for n in names]
m_warm  = [ablation_results[n]['M_warm_mean'] for n in names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Panel 1: M_cold vs M_warm grouped bar
x = np.arange(len(names))
ax1.bar(x-0.2, m_cold, 0.38, label='$M_{\mathrm{cold}}$ (no warmstart)', color='tomato', alpha=0.85)
ax1.bar(x+0.2, m_warm, 0.38, label='$M_{\mathrm{warm}}$ (with warmstart)', color='seagreen', alpha=0.85)
ax1.set_xticks(x); ax1.set_xticklabels(names, rotation=20, ha='right')
ax1.set_ylabel('Samples to reach $\epsilon_{\mathrm{TVD}} < 0.05$')
ax1.set_title('Sample budget: cold vs warm start')
ax1.legend(); ax1.grid(True, alpha=0.3, axis='y')

# Panel 2: speedup ratio
bars = ax2.bar(names, speedups, color='steelblue', alpha=0.85)
ax2.axhline(1.0, color='k', ls='--', lw=1)
for bar, s in zip(bars, speedups):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{s:.1f}x', ha='center', va='bottom', fontsize=9)
ax2.set_ylabel('Speedup ($M_{\mathrm{cold}}/M_{\mathrm{warm}}$)')
ax2.set_title('Warmstart speedup by dataset')
ax2.set_xticklabels(names, rotation=20, ha='right')
ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Variational Warmstart Ablation (Marena 2026, $\epsilon={EPSILON_TARGET}$)', fontsize=11)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR,'warmstart_ablation.pdf')
plt.savefig(path, bbox_inches='tight'); plt.show()
print(f'Saved: {path}')

# LaTeX table
print('\n--- LaTeX table (Section 5.3) ---')
print(r'\begin{tabular}{lccc}')
print(r'\toprule')
print(r'Dataset & $M_{\mathrm{cold}}$ & $M_{\mathrm{warm}}$ & Speedup \\ ')
print(r'\midrule')
for name in names:
    r = ablation_results[name]
    print(f"{name} & {r['M_cold_mean']:.0f}$\pm${r['M_cold_std']:.0f} & {r['M_warm_mean']:.0f}$\pm${r['M_warm_std']:.0f} & {r['speedup_mean']:.1f}\times \\\\")
print(r'\bottomrule'); print(r'\end{tabular}')